# Baseline - sample mean

Dummy prediction to estimate minimum performance

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.special import logit, expit
from scipy.stats import norm
from sklearn.model_selection import GroupShuffleSplit
import seaborn as sns
from sklearn.base import BaseEstimator, RegressorMixin



In [2]:
train_df_all = pd.read_csv("train_augmented.csv",index_col=0)
train_df_all.columns

Index(['category_name', 'geography_name', 'wave_date', 'question_text',
       'qualifying_options', 'disqualifying_options', 'weight_3',
       'base_weight_3', 'percentage_3', 'timeframe_months', 'have_you_quest',
       'are_you_quest', 'multi_choice_quest', 'do_you_own_quest',
       'n_qualifying', 'n_disqualifying', 'qualifying_has_no',
       'qualifying_has_yes', 'continent', 'product_category', 'broad_category',
       'is_luxury', 'is_subscription', 'audience_breadth',
       'purchase_frequency', 'usage_frequency', 'cost_tier'],
      dtype='str')

In [3]:
train_df_all["logit_incidence"] = logit( train_df_all["percentage_3"] )
train_df_all["logit_incidence"].describe()

count    829.000000
mean       0.298250
std        0.995341
min       -3.009467
25%       -0.447312
50%        0.159980
75%        0.942979
max        6.868457
Name: logit_incidence, dtype: float64

## Split dataset into train and validation sets

We want the validation split to have completely new product categories, as will be the case for the test set

In [19]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, val_idx = next(splitter.split(
    train_df_all,
    groups=train_df_all["product_category"]
))

train = train_df_all.iloc[train_idx].copy()
val = train_df_all.iloc[val_idx].copy()

print(f"Train dataset: {len(train)} rows")
print(f"Validation dataset: {len(val)} rows")

Train dataset: 653 rows
Validation dataset: 176 rows


In [28]:
set(train.product_category).intersection(val.product_category)

set()

No product categories are shared between the two sets

In [21]:
target_col = "logit_incidence"

# Dummy model

The prediction is imply the overall mean, and the prediction interval is base on the standard deviation 

In [29]:
y_train = train.logit_incidence
y_val = val.logit_incidence
predict = (~y_val.isna()) * y_train.mean()



## Calculate residuals on the validation set

In [30]:
residuals = y_val - predict
margin_mult = norm.ppf(0.90) 
l80 = predict - y_val.std()*margin_mult
u80 = predict + y_val.std()*margin_mult
in_interval = (y_val>l80) & (y_val<u80)

In [31]:
coverage = in_interval.mean()
print("80% interval coverage:", coverage)

80% interval coverage: 0.7954545454545454


In [32]:
mae = residuals.abs().mean()
print("Mean Absolute Error:", mae)

Mean Absolute Error: 0.7674207598005262
